# Hospital Patient Care Analytics Dashboard

## Interactive Data Engineering and Statistics Dashboard

### Case Study

A multi-specialty hospital collects data from patient registration, appointment scheduling, laboratory reports, wearable health devices, and doctor consultation systems.

This notebook builds a small end-to-end analytics workflow using fictional data. The data is cleaned, integrated, analyzed, and displayed through an interactive dashboard.

### Dashboard goals

- Understand patient flow and appointment status.
- Compare waiting times across departments.
- Review basic laboratory and wearable indicators.
- Identify records that may need additional review.
- Provide an interactive interface for exploring hospital statistics.

> **Important:** The dataset is synthetic. The monitoring logic is illustrative only and is not a clinical diagnosis or a validated medical prediction model.


## 1. Tools Used

- **Pandas:** Data preparation and transformation.
- **NumPy:** Numerical operations.
- **Plotly:** Interactive charts.
- **ipywidgets:** Dashboard filters.
- **SQLite:** Demonstration of local data storage.

The dashboard is designed to run inside Jupyter Notebook or JupyterLab. In Google Colab, the notebook can be executed after installing the required packages if they are not already available.


In [1]:
# Install these packages if they are missing in your environment
# Uncomment the next line when required:
# %pip install pandas numpy plotly ipywidgets

import sqlite3
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

pd.set_option("display.max_columns", None)

print("Dashboard libraries loaded")


Dashboard libraries loaded


## 2. Creating the Source Data

In [2]:
patients = pd.DataFrame({
    "patient_id": range(1001, 1021),
    "age": [24, 67, 45, 72, 33, 58, 81, 39, 52, 28,
            61, 47, 75, 36, 69, 42, 55, 30, 77, 49],
    "gender": ["F", "M", "F", "M", "F", "M", "F", "M", "F", "M",
               "F", "M", "F", "M", "F", "M", "F", "M", "F", "M"],
    "department": [
        "General Medicine", "Cardiology", "General Medicine", "Emergency",
        "Orthopedics", "Cardiology", "Emergency", "General Medicine",
        "Neurology", "Orthopedics", "Cardiology", "General Medicine",
        "Emergency", "Neurology", "Cardiology", "Orthopedics",
        "General Medicine", "Emergency", "Neurology", "General Medicine"
    ]
})

appointments = pd.DataFrame({
    "appointment_id": range(5001, 5021),
    "patient_id": range(1001, 1021),
    "waiting_time_min": [22, 65, 35, 110, 42, 58, 125, 28, 70, 46,
                         62, 31, 98, 55, 74, 39, 26, 115, 88, 33],
    "consultation_time_min": [15, 28, 18, 35, 22, 30, 42, 16, 32, 25,
                              27, 20, 38, 29, 31, 24, 17, 40, 36, 19],
    "appointment_status": [
        "Completed", "Completed", "Completed", "Completed", "Cancelled",
        "Completed", "Completed", "No Show", "Completed", "Completed",
        "Completed", "Completed", "Completed", "Cancelled", "Completed",
        "Completed", "Completed", "Completed", "No Show", "Completed"
    ]
})

labs = pd.DataFrame({
    "patient_id": range(1001, 1021),
    "glucose": [92, 148, 105, 182, 99, 155, 194, 110, 132, 101,
                145, 115, 176, 108, 160, 98, 118, 188, 201, 103],
    "hemoglobin": [14.0, 12.1, 13.2, 10.2, 13.6, 12.0, 9.7, 14.1, 12.8, 13.4,
                   11.8, 13.0, 10.9, 12.6, 11.7, 13.8, 14.2, 10.5, 9.8, 13.5]
})

wearables = pd.DataFrame({
    "patient_id": range(1001, 1021),
    "heart_rate": [74, 102, 81, 116, 79, 98, 121, 83, 105, 80,
                   100, 77, 112, 86, 108, 75, 82, 119, 124, 79],
    "oxygen_saturation": [98, 96, 97, 91, 99, 94, 89, 98, 95, 99,
                          94, 98, 92, 97, 93, 99, 98, 90, 88, 98],
    "temperature": [36.6, 37.1, 36.8, 38.3, 36.5, 37.4, 38.5, 36.7, 37.2, 36.6,
                    37.5, 36.8, 38.0, 36.9, 37.7, 36.5, 36.8, 38.2, 38.6, 36.7]
})

consultations = pd.DataFrame({
    "patient_id": range(1001, 1021),
    "follow_up_required": [
        "No", "Yes", "No", "Yes", "No", "Yes", "Yes", "No", "Yes", "No",
        "Yes", "No", "Yes", "No", "Yes", "No", "No", "Yes", "Yes", "No"
    ],
    "doctor_priority": [
        "Routine", "Medium", "Routine", "High", "Routine", "Medium",
        "High", "Routine", "Medium", "Routine", "Medium", "Routine",
        "High", "Routine", "Medium", "Routine", "Routine", "High",
        "High", "Routine"
    ]
})

print("Source datasets created")


Source datasets created


## 3. Data Integration and Cleaning

In [3]:
hospital_data = (
    patients
    .merge(appointments, on="patient_id", how="left")
    .merge(labs, on="patient_id", how="left")
    .merge(wearables, on="patient_id", how="left")
    .merge(consultations, on="patient_id", how="left")
)

hospital_data = hospital_data.drop_duplicates()
hospital_data["department"] = hospital_data["department"].str.strip().str.title()

numeric_columns = [
    "age", "waiting_time_min", "consultation_time_min",
    "glucose", "hemoglobin", "heart_rate",
    "oxygen_saturation", "temperature"
]

for column in numeric_columns:
    hospital_data[column] = pd.to_numeric(
        hospital_data[column], errors="coerce"
    )

hospital_data["total_visit_time_min"] = (
    hospital_data["waiting_time_min"]
    + hospital_data["consultation_time_min"]
)

print("Combined dataset shape:", hospital_data.shape)
display(hospital_data.head())


Combined dataset shape: (20, 16)


,patient_id,age,gender,department,appointment_id,waiting_time_min,consultation_time_min,appointment_status,glucose,hemoglobin,heart_rate,oxygen_saturation,temperature,follow_up_required,doctor_priority,total_visit_time_min
0,1001,24,F,General Medicine,5001,22,15,Completed,92,14.0,74,98,36.6,No,Routine,37
1,1002,67,M,Cardiology,5002,65,28,Completed,148,12.1,102,96,37.1,Yes,Medium,93
2,1003,45,F,General Medicine,5003,35,18,Completed,105,13.2,81,97,36.8,No,Routine,53
3,1004,72,M,Emergency,5004,110,35,Completed,182,10.2,116,91,38.3,Yes,High,145
4,1005,33,F,Orthopedics,5005,42,22,Cancelled,99,13.6,79,99,36.5,No,Routine,64


## 4. Creating an Illustrative Monitoring Score

In [4]:
# This is a classroom demonstration, not a medical prediction model.
hospital_data["monitoring_score"] = (
    (hospital_data["heart_rate"] > 105).astype(int)
    + (hospital_data["oxygen_saturation"] < 94).astype(int)
    + (hospital_data["temperature"] >= 38).astype(int)
    + (hospital_data["glucose"] > 140).astype(int)
    + (hospital_data["hemoglobin"] < 11).astype(int)
    + (hospital_data["doctor_priority"] == "High").astype(int)
)

hospital_data["monitoring_category"] = pd.cut(
    hospital_data["monitoring_score"],
    bins=[-1, 1, 3, 6],
    labels=["Low", "Medium", "High"]
)

hospital_data[[
    "patient_id", "department", "monitoring_score",
    "monitoring_category", "follow_up_required"
]].head(10)


,patient_id,department,monitoring_score,monitoring_category,follow_up_required
0,1001,General Medicine,0,Low,No
1,1002,Cardiology,1,Low,Yes
2,1003,General Medicine,0,Low,No
3,1004,Emergency,6,High,Yes
4,1005,Orthopedics,0,Low,No
5,1006,Cardiology,1,Low,Yes
6,1007,Emergency,6,High,Yes
7,1008,General Medicine,0,Low,No
8,1009,Neurology,0,Low,Yes
9,1010,Orthopedics,0,Low,No


## 5. Data Quality Summary

In [5]:
quality_summary = pd.DataFrame({
    "Metric": [
        "Total records",
        "Duplicate rows",
        "Missing values",
        "Unique patients",
        "Unique departments"
    ],
    "Value": [
        len(hospital_data),
        int(hospital_data.duplicated().sum()),
        int(hospital_data.isna().sum().sum()),
        hospital_data["patient_id"].nunique(),
        hospital_data["department"].nunique()
    ]
})

display(quality_summary)


,Metric,Value
0,Total records,20
1,Duplicate rows,0
2,Missing values,0
3,Unique patients,20
4,Unique departments,5


## 6. Dashboard Preparation

In [6]:
def make_kpi_card(title, value, subtitle=""):
    return f'''
    <div style="
        background:#f8fafc;
        border:1px solid #e2e8f0;
        border-radius:14px;
        padding:18px;
        width:210px;
        min-height:100px;
        box-shadow:0 2px 6px rgba(15,23,42,0.05);
        font-family:Arial,sans-serif;
    ">
        <div style="font-size:13px;color:#64748b;margin-bottom:8px;">{title}</div>
        <div style="font-size:30px;font-weight:700;color:#0f172a;">{value}</div>
        <div style="font-size:12px;color:#64748b;margin-top:6px;">{subtitle}</div>
    </div>
    '''

def show_kpis(data):
    total_patients = data["patient_id"].nunique()
    average_wait = data["waiting_time_min"].mean() if len(data) else 0
    completed = (data["appointment_status"] == "Completed").sum()
    flagged = (data["monitoring_category"] == "High").sum()

    cards = [
        make_kpi_card("TOTAL PATIENTS", total_patients, "Filtered records"),
        make_kpi_card("AVG WAITING", f"{average_wait:.1f} min", "Average service wait"),
        make_kpi_card("COMPLETED VISITS", completed, "Appointment status"),
        make_kpi_card("HIGH MONITORING", flagged, "Illustrative screening")
    ]

    display(HTML(
        '<div style="display:flex;flex-wrap:wrap;gap:14px;margin:10px 0 22px;">'
        + "".join(cards)
        + "</div>"
    ))


## 7. Interactive Dashboard

In [7]:
department_options = ["All Departments"] + sorted(
    hospital_data["department"].dropna().unique().tolist()
)

status_options = ["All Statuses"] + sorted(
    hospital_data["appointment_status"].dropna().unique().tolist()
)

department_filter = widgets.Dropdown(
    options=department_options,
    value="All Departments",
    description="Department:",
    layout=widgets.Layout(width="330px")
)

status_filter = widgets.Dropdown(
    options=status_options,
    value="All Statuses",
    description="Status:",
    layout=widgets.Layout(width="330px")
)

min_wait_filter = widgets.IntSlider(
    value=0,
    min=0,
    max=int(hospital_data["waiting_time_min"].max()),
    step=5,
    description="Min wait:",
    continuous_update=False,
    layout=widgets.Layout(width="330px")
)

reset_button = widgets.Button(
    description="Reset Filters",
    button_style="info",
    icon="refresh"
)

dashboard_output = widgets.Output()

def get_filtered_data():
    filtered = hospital_data.copy()

    if department_filter.value != "All Departments":
        filtered = filtered[
            filtered["department"] == department_filter.value
        ]

    if status_filter.value != "All Statuses":
        filtered = filtered[
            filtered["appointment_status"] == status_filter.value
        ]

    filtered = filtered[
        filtered["waiting_time_min"] >= min_wait_filter.value
    ]

    return filtered

def draw_dashboard(*args):
    with dashboard_output:
        clear_output(wait=True)

        filtered = get_filtered_data()

        display(HTML(
            "<h2 style='font-family:Arial;color:#0f172a;'>Hospital Statistics Dashboard</h2>"
            "<p style='font-family:Arial;color:#64748b;'>"
            "Use the filters to explore the synthetic hospital dataset."
            "</p>"
        ))

        show_kpis(filtered)

        if filtered.empty:
            display(HTML("<b>No records match the selected filters.</b>"))
            return

        department_chart = (
            filtered.groupby("department", as_index=False)
            .agg(
                average_wait=("waiting_time_min", "mean"),
                patient_count=("patient_id", "count")
            )
        )

        fig1 = px.bar(
            department_chart,
            x="department",
            y="average_wait",
            text_auto=".1f",
            title="Average Waiting Time by Department",
            labels={
                "department": "Department",
                "average_wait": "Average Waiting Time (minutes)"
            },
            template="plotly_white"
        )
        fig1.update_layout(
            height=420,
            title_x=0.03,
            xaxis_tickangle=-30
        )
        fig1.show()

        status_chart = (
            filtered["appointment_status"]
            .value_counts()
            .rename_axis("status")
            .reset_index(name="count")
        )

        fig2 = px.pie(
            status_chart,
            names="status",
            values="count",
            hole=0.45,
            title="Appointment Status Distribution",
            template="plotly_white"
        )
        fig2.update_layout(height=400, title_x=0.03)
        fig2.show()

        risk_chart = (
            filtered["monitoring_category"]
            .value_counts()
            .reindex(["Low", "Medium", "High"])
            .fillna(0)
            .rename_axis("category")
            .reset_index(name="count")
        )

        fig3 = px.bar(
            risk_chart,
            x="category",
            y="count",
            text="count",
            title="Illustrative Monitoring Categories",
            labels={"category": "Category", "count": "Records"},
            template="plotly_white"
        )
        fig3.update_layout(height=380, title_x=0.03)
        fig3.show()

        display(HTML("<h3 style='font-family:Arial;color:#0f172a;'>Filtered Patient Records</h3>"))

        display(
            filtered[[
                "patient_id", "department", "appointment_status",
                "waiting_time_min", "monitoring_score",
                "monitoring_category", "follow_up_required"
            ]].sort_values("waiting_time_min", ascending=False)
        )

def reset_filters(button):
    department_filter.value = "All Departments"
    status_filter.value = "All Statuses"
    min_wait_filter.value = 0

department_filter.observe(draw_dashboard, names="value")
status_filter.observe(draw_dashboard, names="value")
min_wait_filter.observe(draw_dashboard, names="value")
reset_button.on_click(reset_filters)

controls = widgets.HBox(
    [department_filter, status_filter, min_wait_filter, reset_button],
    layout=widgets.Layout(
        flex_flow="row wrap",
        gap="12px",
        align_items="center"
    )
)

display(controls)
display(dashboard_output)

draw_dashboard()


Output()

## 8. Department Workload Analysis

In [8]:
department_workload = (
    hospital_data.groupby("department", as_index=False)
    .agg(
        total_visits=("patient_id", "count"),
        average_waiting_time=("waiting_time_min", "mean"),
        average_consultation_time=("consultation_time_min", "mean"),
        average_total_visit_time=("total_visit_time_min", "mean")
    )
    .sort_values("average_waiting_time", ascending=False)
)

display(department_workload)

fig = px.scatter(
    department_workload,
    x="average_waiting_time",
    y="total_visits",
    size="average_total_visit_time",
    text="department",
    title="Department Workload and Waiting Time",
    labels={
        "average_waiting_time": "Average Waiting Time (minutes)",
        "total_visits": "Total Visits"
    },
    template="plotly_white"
)

fig.update_traces(textposition="top center")
fig.update_layout(height=500)
fig.show()


,department,total_visits,average_waiting_time,average_consultation_time,average_total_visit_time
1,Emergency,4,112.000000,38.750000,150.750000
3,Neurology,3,71.000000,32.333333,103.333333
0,Cardiology,4,64.750000,29.000000,93.750000
4,Orthopedics,3,42.333333,23.666667,66.000000
2,General Medicine,6,29.166667,17.500000,46.666667


## 9. Age Distribution and Patient Demographics

In [9]:
fig = px.histogram(
    hospital_data,
    x="age",
    nbins=8,
    color="gender",
    barmode="overlay",
    opacity=0.75,
    title="Patient Age Distribution",
    labels={"age": "Age", "count": "Number of Patients"},
    template="plotly_white"
)

fig.update_layout(height=430)
fig.show()


## 10. Laboratory and Wearable Statistics

In [10]:
health_summary = hospital_data[[
    "glucose", "hemoglobin", "heart_rate",
    "oxygen_saturation", "temperature"
]].describe().T.reset_index()

health_summary = health_summary.rename(columns={"index": "metric"})
display(health_summary)


,metric,count,mean,std,min,25%,50%,75%,max
0,glucose,20.0,136.500,36.585157,92.0,104.50,125.0,164.000,201.0
1,hemoglobin,20.0,12.345,1.474155,9.7,11.50,12.7,13.525,14.2
2,heart_rate,20.0,95.050,17.270221,74.0,79.75,92.0,109.000,124.0
3,oxygen_saturation,20.0,95.150,3.572924,88.0,92.75,96.5,98.000,99.0
4,temperature,20.0,37.270,0.709410,36.5,36.70,37.0,37.775,38.6


In [11]:
fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=("Glucose Distribution", "Oxygen Saturation Distribution")
)

fig.add_trace(
    go.Histogram(
        x=hospital_data["glucose"],
        name="Glucose",
        nbinsx=8
    ),
    row=1,
    col=1
)

fig.add_trace(
    go.Histogram(
        x=hospital_data["oxygen_saturation"],
        name="Oxygen Saturation",
        nbinsx=8
    ),
    row=1,
    col=2
)

fig.update_layout(
    title_text="Health Indicator Distributions",
    height=430,
    template="plotly_white",
    showlegend=False
)

fig.show()


## 11. Exporting Filtered Data

In [12]:
export_data = hospital_data[[
    "patient_id", "department", "appointment_status",
    "waiting_time_min", "consultation_time_min",
    "monitoring_score", "monitoring_category",
    "follow_up_required"
]].copy()

export_path = Path("hospital_dashboard_analytics.csv")
export_data.to_csv(export_path, index=False)

print("Exported:", export_path)


Exported: hospital_dashboard_analytics.csv


## 12. Optional SQLite Storage

In [13]:
database_path = "hospital_dashboard.db"
connection = sqlite3.connect(database_path)

hospital_data.to_sql(
    "hospital_analytics",
    connection,
    if_exists="replace",
    index=False
)

stored_count = pd.read_sql_query(
    "SELECT COUNT(*) AS record_count FROM hospital_analytics",
    connection
)

display(stored_count)

connection.close()
print("SQLite storage demonstration completed")


,record_count
0,20


SQLite storage demonstration completed


## 13. Project Summary

This notebook developed an interactive hospital statistics dashboard.

### Implemented components

- Multiple synthetic hospital source datasets.
- Data integration using patient identifiers.
- Basic cleaning and numeric conversion.
- An illustrative monitoring score.
- Interactive department, status, and waiting-time filters.
- KPI cards.
- Interactive Plotly charts.
- Department workload analysis.
- Demographic and health-indicator visualizations.
- CSV export and SQLite storage.

### Limitations

The data is fictional and small. The monitoring score is not clinically validated. A real hospital application would require secure data ingestion, access controls, audit logs, clinical validation, monitoring, and appropriate governance.

### Possible future improvements

- Connect the dashboard to a real database.
- Add date-based filtering.
- Add automated data-quality alerts.
- Use a workflow scheduler for regular pipeline execution.
- Publish the dashboard through Streamlit or another web application framework.
